In [ ]:
import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt
from tqdm import tqdm

# -----------------------------
# SETTINGS
# -----------------------------
MODEL_PATH = './flaubert_informel'  # Path to your fine-tuned STS model
INPUT_FILE = 'sts_finetuning_dataset.csv'  # File containing sentence1 / sentence2
OUTPUT_FILE = 'sts_data_eval_final.csv'   # Output file with annotated scores
MAX_LENGTH = 128

# -----------------------------
# LOAD MODEL + TOKENIZER
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
model.to(device)
model.eval()

# -----------------------------
# LOAD DATASET
# -----------------------------
print("Loading dataset...")
df = pd.read_csv(INPUT_FILE, sep=";", on_bad_lines='skip', encoding="utf-8")

# Ensure the necessary columns exist and are cleaned
df.columns = df.columns.str.lower().str.strip()

if not {'sentence1', 'sentence2', 'similarity_score'}.issubset(df.columns):
    raise ValueError("Dataset must contain 'sentence1', 'sentence2' and 'similarity_score' columns")

# Remove empty or null sentence pairs
df = df.dropna(subset=['sentence1', 'sentence2'])

# Convert sentences to strings
df['sentence1'] = df['sentence1'].astype(str)
df['sentence2'] = df['sentence2'].astype(str)

# -----------------------------
# FUNCTION TO PREDICT STS SCORE
# -----------------------------
def predict_sts_score(s1, s2):
    inputs = tokenizer(
        s1, s2,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        score = outputs.logits.squeeze().item()

    # Clamp the score between 0 and 5
    return max(0, min(5, score))

# -----------------------------
# FUNCTION TO REMOVE DUPLICATE AND INVERSE PAIRS
# -----------------------------
def remove_duplicate_and_inverse_pairs(df):
    seen_pairs = set()
    clean_pairs = []

    for index, row in df.iterrows():
        pair1 = (row['sentence1'], row['sentence2'])
        pair2 = (row['sentence2'], row['sentence1'])
        
        # Check if pair is duplicate or inverse pair
        if pair1 not in seen_pairs and pair2 not in seen_pairs:
            seen_pairs.add(pair1)
            clean_pairs.append(row)
    
    clean_df = pd.DataFrame(clean_pairs)
    clean_df.reset_index(drop=True, inplace=True)  # Reset index after cleaning
    return clean_df

# -----------------------------
# PROCESSING DATA IN BATCHES FOR EFFICIENCY
# -----------------------------
def batch_predict_sts_scores(df, batch_size=16):
    predictions = []
    
    # Process data in batches
    for i in tqdm(range(0, len(df), batch_size)):
        batch = df.iloc[i:i + batch_size]
        
        # Tokenize the batch
        inputs = tokenizer(
            batch["sentence1"].tolist(),
            batch["sentence2"].tolist(),
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=MAX_LENGTH
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            scores = outputs.logits.squeeze(-1).cpu().numpy().astype(float).ravel()
            predictions.extend(scores.tolist())

    return predictions

# -----------------------------
# PREDICT ALL SCORES IN BATCHES
# -----------------------------
print("Generating STS scores in batches...")

# Clean dataset by removing duplicate and inverse pairs
df_clean = remove_duplicate_and_inverse_pairs(df)

# Predict scores for all pairs
predicted_scores = batch_predict_sts_scores(df_clean)
predicted_scores = np.asarray(predicted_scores, dtype=float)

# Save the results back to the dataframe
df_clean["similarity_score"] = np.clip(predicted_scores, 0.0, 5.0)

# -----------------------------
# EVALUATION AND PERFORMANCE METRICS
# -----------------------------
# Calculate MSE, MAE, RMSE, and R2
true_scores = df_clean['similarity_score'].tolist()
predicted_scores_array = np.array(predicted_scores)
true_scores_array = np.array(true_scores)

mse = mean_squared_error(true_scores_array, predicted_scores_array)
mae = mean_absolute_error(true_scores_array, predicted_scores_array)
rmse = np.sqrt(mse)
r2 = r2_score(true_scores_array, predicted_scores_array)

# Calculate Pearson and Spearman Correlations
spearman_corr, _ = spearmanr(true_scores_array, predicted_scores_array)
pearson_corr, _ = pearsonr(true_scores_array, predicted_scores_array)

# -----------------------------
# PRINT EVALUATION RESULTS
# -----------------------------
print(f"\nMSE: {mse:.4f}, MAE: {mae:.4f}, RMSE: {rmse:.4f}, R²: {r2:.4f}")
print(f"Spearman Correlation: {spearman_corr:.4f}, Pearson Correlation: {pearson_corr:.4f}")

# -----------------------------
# SAVE RESULTS WITH PROPER FORMATTING
# -----------------------------
# Keep only the three columns and use semicolon as separator
df_clean[["sentence1", "sentence2", "similarity_score"]].to_csv(OUTPUT_FILE, sep=";", index=False, encoding="utf-8")

print(f"\nResults saved to {OUTPUT_FILE}")

Loading model and tokenizer...
Loading dataset...
Generating STS scores in batches...


  3%|▎         | 17/590 [00:02<00:43, 13.05it/s]Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
 14%|█▍        | 83/590 [00:06<00:35, 14.13it/s]Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.



MSE: 0.0005, MAE: 0.0025, RMSE: 0.0226, R²: 0.9985
Spearman Correlation: 1.0000, Pearson Correlation: 0.9994

Results saved to sts_data_eval_final.csv


In [ ]:
import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt

# -----------------------------
# SETTINGS
# -----------------------------
MODEL_PATH = 'flaubert/flaubert_base_cased'  # Flaubert base model for baseline evaluation
INPUT_FILE = 'STS_dataset.csv'  # File containing the 500 annotated sentence pairs
OUTPUT_FILE = 'flaubert_base_evaluation_results.csv'  # Output file for annotated scores
MAX_LENGTH = 128

# -----------------------------
# LOAD MODEL + TOKENIZER
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
model.to(device)
model.eval()

# -----------------------------
# LOAD DATASET
# -----------------------------
print("Loading dataset...")
df = pd.read_csv(INPUT_FILE, sep=";", on_bad_lines='skip', encoding="utf-8")

# Ensure the necessary columns exist and are cleaned
df.columns = df.columns.str.lower().str.strip()

if not {'sentence1', 'sentence2', 'similarity_score'}.issubset(df.columns):
    raise ValueError("Dataset must contain 'sentence1', 'sentence2' and 'similarity_score' columns")

# Remove empty or null sentence pairs
df = df.dropna(subset=['sentence1', 'sentence2'])

# Convert sentences to strings
df['sentence1'] = df['sentence1'].astype(str)
df['sentence2'] = df['sentence2'].astype(str)

# -----------------------------
# FUNCTION TO PREDICT STS SCORE
# -----------------------------
def predict_sts_score(s1, s2):
    inputs = tokenizer(
        s1, s2,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        score = outputs.logits.squeeze().item()

    # Clamp the score between 0 and 5
    return max(0, min(5, score))

# -----------------------------
# FUNCTION TO REMOVE DUPLICATE AND INVERSE PAIRS
# -----------------------------
def remove_duplicate_and_inverse_pairs(df):
    seen_pairs = set()
    clean_pairs = []

    for index, row in df.iterrows():
        pair1 = (row['sentence1'], row['sentence2'])
        pair2 = (row['sentence2'], row['sentence1'])
        
        # Check if pair is duplicate or inverse pair
        if pair1 not in seen_pairs and pair2 not in seen_pairs:
            seen_pairs.add(pair1)
            clean_pairs.append(row)
    
    clean_df = pd.DataFrame(clean_pairs)
    clean_df.reset_index(drop=True, inplace=True)  # Reset index after cleaning
    return clean_df

# -----------------------------
# PREDICT ALL SCORES IN BATCHES
# -----------------------------
print("Generating STS scores in batches...")

# Clean dataset by removing duplicate and inverse pairs
df_clean = remove_duplicate_and_inverse_pairs(df)

predicted_scores = []
for index, row in df_clean.iterrows():
    try:
        score = predict_sts_score(row['sentence1'], row['sentence2'])
        predicted_scores.append(score)
    except Exception as e:
        print(f"Error processing pair {index}: {e}")
        predicted_scores.append(0)  # Default to 0 if there's an error

# -----------------------------
# EVALUATION AND PERFORMANCE METRICS
# -----------------------------
# Calculate MSE, MAE, RMSE, and R2
true_scores = df_clean['similarity_score'].tolist()
predicted_scores_array = np.array(predicted_scores)
true_scores_array = np.array(true_scores)

mse = mean_squared_error(true_scores_array, predicted_scores_array)
mae = mean_absolute_error(true_scores_array, predicted_scores_array)
rmse = np.sqrt(mse)
r2 = r2_score(true_scores_array, predicted_scores_array)

# Calculate Pearson and Spearman Correlations
spearman_corr, _ = spearmanr(true_scores_array, predicted_scores_array)
pearson_corr, _ = pearsonr(true_scores_array, predicted_scores_array)

# -----------------------------
# PRINT EVALUATION RESULTS
# -----------------------------
print(f"\nMSE: {mse:.4f}, MAE: {mae:.4f}, RMSE: {rmse:.4f}, R²: {r2:.4f}")
print(f"Spearman Correlation: {spearman_corr:.4f}, Pearson Correlation: {pearson_corr:.4f}")

# -----------------------------
# SAVE RESULTS WITH PROPER FORMATTING
# -----------------------------
# Keep only the three columns and use semicolon as separator
df_clean[["sentence1", "sentence2", "similarity_score"]].to_csv(OUTPUT_FILE, sep=";", index=False, encoding="utf-8")

print(f"\nResults saved to {OUTPUT_FILE}")
